Лабораторнгая работа №1 СИИиМО

Введение: 

Цель работы: изучение основ линейной регрессии, построение простейших моделей регрессии, обучение на реальных данных и оценка качества. 
Задачи:
1. загрузить датасет
2. подготовить данные для дальнейшей работы (провести первичный анализ, визуализировать распределение признаков и целевой переменной)
3. Провести предобработку данных
4. Построить матрицу корреляций. Сделать выводы о наличии мультиколлинеарности (расчет VIF-коэффициента)
5. Построить регрессионные модели (линейная, лассо и гребневая). Если целевая переменная - категориальная, то исследовать логистическую регрессию. Разделить на тренировочную и тестовую выборки (80/20 или 70/30). Использовать кросс-валидацию. Оценить качество построенной модели с помощью метрик: RMSE (Root Mean Square Error), R² (коэффициент детерминации) и MAPE (Mean Absolute Percentage Error).
6. Устранить мультиколлинеарность и снизить размерность признаков с помощью метода главных компонент (PCA). Перед проведением PCA провести стандартизацию данных.
7. Повторить шаг 5 (линейная, лассо и гребневая регрессия), но использовать в качестве признаков не исходные данные, а главные компоненты. Сравнить метрики качества (RMSE, R² и MAPE) моделей, обученных на исходных данных и на главных компонентах.

Датасет:

Столбцы:

date — дата наблюдения;
adj close — скорректированная цена закрытия;
close — цена закрытия (целевая переменная);
high, low, open — максимальная, минимальная и цена открытия;
volume — объём торгов;
ma_7, ma_30, ma_90 — скользящие средние;
daily_return — дневная доходность;
volatility_7, volatility_30 — волатильность;
rsi — индекс относительной силы;
macd, macd_signal — индикаторы MACD;
bb_upper, bb_lower — границы полос Боллинджера.

Этот набор данных содержит исторические данные о ежедневных ценах на золото, а также множество технических индикаторов, используемых для анализа и прогнозирования временных рядов. Он включает в себя основные рыночные показатели, такие как цена открытия, цена закрытия, максимальная и минимальная цены за день, скорректированная цена закрытия и объем торгов. Помимо необработанных данных о ценах, в наборе данных представлены расчетные показатели, такие как скользящие средние (за 7, 30 и 90 дней), дневная доходность, краткосрочная и долгосрочная волатильность, индекс относительной силы (RSI), MACD и его сигнальная линия, а также полосы Боллинджера (верхняя и нижняя). Благодаря этим расчетным показателям набор данных отлично подходит для моделей машинного обучения, особенно для задач прогнозирования цен и анализа трендов.

Все признаки, кроме date, — некатегориальные (числовые);
adj close и ma_* являются производными от close → утечка данных, их нужно исключить;
Признаки сильно коррелируют - ожидается мультиколлинеарность.

Подготовка данных:

Процедуры предобработки:

Загрузка CSV-файла.
Проверка типов данных и пропусков.
Отбор только некатегориальных (числовых) столбцов.
Исключение утечки данных: adj close, ma_7, ma_30, ma_90.
Разделение на train/test в пропорции 80/20 (с shuffle=False для сохранения временной структуры).
Стандартизация (StandardScaler) — только перед PCA.
Пропуски: отсутствуют.
Категориальные переменные: отсутствуют.
Нормализация: выполнена через StandardScaler перед PCA.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.decomposition import PCA
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_percentage_error
from statsmodels.stats.outliers_influence import variance_inflation_factor
import warnings
warnings.filterwarnings('ignore')

# Настройка отображения графиков
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

# Настройка отображения pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

print("Библиотеки успешно загружены")

Загружаем CSV-файл с данными о ценах на золото. Используем `pd.read_csv()` с параметром `parse_dates=['date']` для автоматического преобразования столбца с датами в формат datetime. Это необходимо для корректной работы с временными рядами и визуализации.

In [ ]:
# Загрузка данных
df = pd.read_csv('gold_price_dts.csv', parse_dates=['date'])

print(f"Размер датасета: {df.shape}")
print(f"Количество наблюдений: {df.shape[0]}")
print(f"Количество признаков: {df.shape[1]}")
print("\nПервые 5 строк:")
df.head()